# STRUM Dataset Preprocessing

## Objective

Prepare the STRUM Sentinel-2 dataset for:

- Baseline U-Net
- Hybrid SAM-U-Net

Steps:

1. Create train/validation/test split
2. Extract RGB bands
3. Normalize images
4. Convert masks to binary
5. Save processed dataset

In [18]:
from pathlib import Path

import numpy as np
import pandas as pd

import rasterio

import matplotlib.pyplot as plt

from tqdm import tqdm

from sklearn.model_selection import train_test_split

import os

## Dataset Paths

In [19]:
ROOT = Path("../dataset/STRUM")

DATASET_PATH = (
    ROOT /
    "Dataset" /
    "Dataset"
)

S2_PATH = (
    DATASET_PATH /
    "Sentinel2" /
    "S2"
)

MASK_PATH = (
    DATASET_PATH /
    "Sentinel2" /
    "Floodmaps"
)

PROCESSED_PATH = Path(
    "../dataset/processed/STRUM"
)

In [20]:
print(S2_PATH.exists())
print(MASK_PATH.exists())

True
True


## Verify folders for processed data 

In [21]:
for split in [
    "train",
    "val",
    "test"
]:
    
    print(
        PROCESSED_PATH /
        split /
        "images"
    )

..\dataset\processed\STRUM\train\images
..\dataset\processed\STRUM\val\images
..\dataset\processed\STRUM\test\images


## Load Image and Mask Files

In [22]:
image_files = sorted(
    list(S2_PATH.glob("*.tif"))
)

mask_files = sorted(
    list(MASK_PATH.glob("*.tif"))
)

print(
    "Images:",
    len(image_files)
)

print(
    "Masks:",
    len(mask_files)
)

Images: 2675
Masks: 2675


In [23]:
image_names = {
    f.stem: f
    for f in image_files
}

mask_names = {
    f.stem: f
    for f in mask_files
}

In [24]:
common_ids = sorted(
    list(
        set(image_names.keys())
        &
        set(mask_names.keys())
    )
)

print(
    "Matched Pairs:",
    len(common_ids)
)

Matched Pairs: 2675


## Create Paper-Compliant Split

80%
10%
10%

In [25]:
train_ids, temp_ids = train_test_split(
    common_ids,
    test_size=0.2,
    random_state=42,
    shuffle=True
)

In [26]:
val_ids, test_ids = train_test_split(
    temp_ids,
    test_size=0.5,
    random_state=42,
    shuffle=True
)

In [27]:
print(
    "Train:",
    len(train_ids)
)

print(
    "Val:",
    len(val_ids)
)

print(
    "Test:",
    len(test_ids)
)

Train: 2140
Val: 267
Test: 268


## RGB Extraction

The original STRUM Sentinel-2 tiles contain 9 bands:

B02, B03, B04, B05, B06, B07, B08, B11, B12

For this project we only use:

- B04 (Red)
- B03 (Green)
- B02 (Blue)

to construct RGB imagery compatible with SAM.

In [28]:
RGB_INDICES = {
    "red": 2,
    "green": 1,
    "blue": 0
}

In [33]:
def extract_rgb(image):

    red = image[RGB_INDICES["red"]]
    green = image[RGB_INDICES["green"]]
    blue = image[RGB_INDICES["blue"]]

    rgb = np.stack(
        [red, green, blue],
        axis=0
    )

    return rgb

## Image Normalization

The released STRUM tiles are already scaled and do not contain raw Sentinel-2 reflectance values.

Therefore, min-max normalization is applied per image to map values into the range [0,1].

In [37]:
def normalize_image(image):

    image = np.nan_to_num(
        image,
        nan=0.0,
        posinf=0.0,
        neginf=0.0
    )

    image_min = image.min()
    image_max = image.max()

    if image_max == image_min:

        return np.zeros_like(
            image,
            dtype=np.float32
        )

    image = (
        image - image_min
    ) / (
        image_max - image_min
    )

    return image.astype(
        np.float32
    )

In [50]:
sample_id = train_ids[0]

image_path = image_names[sample_id]

with rasterio.open(image_path) as src:
    image = src.read()

In [51]:
rgb = extract_rgb(image)

rgb_norm = normalize_image(rgb)

print("Before Normalization")
print("Min:", rgb.min())
print("Max:", rgb.max())

print("\nAfter Normalization")
print("Min:", rgb_norm.min())
print("Max:", rgb_norm.max())

Before Normalization
Min: 0.20333333333333334
Max: 1.5466666666666666

After Normalization
Min: 0.0
Max: 1.0


## Binary Mask Conversion

Original mask classes:

0 = Non-Water

1 = Water

2 = Water

For flood segmentation we use:

0 = Non-Water

1 = Water

In [52]:
def convert_to_binary(mask):

    mask_binary = np.where(
        mask > 0,
        1,
        0
    )

    return mask_binary.astype(np.uint8)

In [53]:
all_classes = set()

for mask_file in mask_files:

    with rasterio.open(mask_file) as src:
        mask = src.read(1)

    all_classes.update(np.unique(mask))

print(sorted(all_classes))

[np.int32(0), np.int32(1), np.int32(2), np.int32(3), np.int32(4), np.int32(5)]


## Mask Analysis

The STRUM dataset contains six original classes:

- 0 : Background / AOI
- 1 : Flooded Area
- 2 : River
- 3 : Open Water
- 4 : Reservoir
- 5 : Lake

For flood segmentation, all water-related classes were merged into a single Water class.

Binary Mapping:

- 0 → Non-Water
- 1,2,3,4,5 → Water

In [54]:
from collections import Counter

class_counter = Counter()

for mask_file in mask_files:

    with rasterio.open(mask_file) as src:
        mask = src.read(1)

    values, counts = np.unique(mask, return_counts=True)

    for v, c in zip(values, counts):
        class_counter[v] += c

print(class_counter)

Counter({np.int32(0): np.int64(31739964), np.int32(1): np.int64(9865084), np.int32(3): np.int64(853493), np.int32(5): np.int64(656571), np.int32(2): np.int64(570702), np.int32(4): np.int64(141386)})


### Dataset Class Distribution

Pixel statistics across the entire STRUM Sentinel-2 dataset:

| Class | Percentage |
|---------|---------|
| 0 (Background) | 72.42% |
| 1 (Flooded Area) | 22.51% |
| 2 (River) | 1.30% |
| 3 (Open Water) | 1.95% |
| 4 (Reservoir) | 0.32% |
| 5 (Lake) | 1.50% |

After binary remapping:

- Non-Water: 72.42%
- Water: 27.58%

The dataset therefore exhibits class imbalance, motivating the use of Dice Score, IoU, Precision, Recall, and F1 Score in addition to Accuracy.

In [55]:
rgb = extract_rgb(image)

rgb_norm = normalize_image(rgb)

print(rgb_norm.min())
print(rgb_norm.max())

0.0
1.0


In [56]:
sample_id = train_ids[0]

image_path = image_names[sample_id]
mask_path = mask_names[sample_id]

with rasterio.open(image_path) as src:
    image = src.read()

with rasterio.open(mask_path) as src:
    mask = src.read(1)

rgb = extract_rgb(image)

rgb_norm = normalize_image(rgb)

mask_binary = convert_to_binary(mask)

print("RGB Shape:", rgb_norm.shape)
print("Mask Shape:", mask_binary.shape)

print("RGB Range:",
      rgb_norm.min(),
      rgb_norm.max())

print("Mask Classes:",
      np.unique(mask_binary))

RGB Shape: (3, 128, 128)
Mask Shape: (128, 128)
RGB Range: 0.0 1.0
Mask Classes: [0 1]


### Process and Save Entire Dataset

The complete STRUM dataset is processed and saved into:

dataset/processed/STRUM/

train/
val/
test/

Each image is stored as:

(3,128,128)

Each mask is stored as:

(128,128)

with binary classes:

0 = Non-Water
1 = Water

In [57]:
def process_and_save(split_ids, split_name):

    image_output_dir = (
        PROCESSED_PATH /
        split_name /
        "images"
    )

    mask_output_dir = (
        PROCESSED_PATH /
        split_name /
        "masks"
    )

    for sample_id in tqdm(
        split_ids,
        desc=f"Processing {split_name}"
    ):

        image_path = image_names[sample_id]
        mask_path = mask_names[sample_id]

        # --------------------
        # Load image
        # --------------------

        with rasterio.open(image_path) as src:
            image = src.read()

        # --------------------
        # Load mask
        # --------------------

        with rasterio.open(mask_path) as src:
            mask = src.read(1)

        # --------------------
        # RGB Extraction
        # --------------------

        image = extract_rgb(image)

        # --------------------
        # Normalization
        # --------------------

        image = normalize_image(image)

        # --------------------
        # Binary Conversion
        # --------------------

        mask = convert_to_binary(mask)

        # --------------------
        # Save
        # --------------------

        np.save(
            image_output_dir /
            f"{sample_id}.npy",
            image
        )

        np.save(
            mask_output_dir /
            f"{sample_id}.npy",
            mask
        )

In [58]:
process_and_save(
    train_ids,
    "train"
)

Processing train: 100%|██████████| 2140/2140 [01:20<00:00, 26.65it/s]


In [59]:
process_and_save(
    val_ids,
    "val"
)

Processing val: 100%|██████████| 267/267 [00:15<00:00, 17.37it/s]


In [60]:
process_and_save(
    test_ids,
    "test"
)

Processing test: 100%|██████████| 268/268 [00:15<00:00, 17.28it/s]


In [61]:
train_images = list(
    (
        PROCESSED_PATH /
        "train" /
        "images"
    ).glob("*.npy")
)

train_masks = list(
    (
        PROCESSED_PATH /
        "train" /
        "masks"
    ).glob("*.npy")
)

print("Train Images:", len(train_images))
print("Train Masks :", len(train_masks))

Train Images: 2140
Train Masks : 2140


In [62]:
for split in [
    "train",
    "val",
    "test"
]:

    images = list(
        (
            PROCESSED_PATH /
            split /
            "images"
        ).glob("*.npy")
    )

    masks = list(
        (
            PROCESSED_PATH /
            split /
            "masks"
        ).glob("*.npy")
    )

    print(
        f"{split.upper()}"
    )

    print(
        "Images:",
        len(images)
    )

    print(
        "Masks :",
        len(masks)
    )

    print("-"*30)

TRAIN
Images: 2140
Masks : 2140
------------------------------
VAL
Images: 267
Masks : 267
------------------------------
TEST
Images: 268
Masks : 268
------------------------------


In [63]:
sample_image = np.load(
    train_images[0]
)

sample_mask = np.load(
    train_masks[0]
)

print(
    "Image Shape:",
    sample_image.shape
)

print(
    "Mask Shape:",
    sample_mask.shape
)

print(
    "Image Range:",
    sample_image.min(),
    sample_image.max()
)

print(
    "Mask Classes:",
    np.unique(sample_mask)
)

Image Shape: (3, 128, 128)
Mask Shape: (128, 128)
Image Range: 0.0 1.0
Mask Classes: [0 1]


## Preprocessing Summary

The STRUM Sentinel-2 dataset was successfully prepared for model training.

Processing steps:

- Random 80/10/10 split
- RGB extraction (B04, B03, B02)
- Min-max normalization
- Binary mask conversion
- Saved as NumPy arrays

Final dataset:

| Split | Samples |
|---------|---------:|
| Train | 2140 |
| Validation | 267 |
| Test | 268 |

Image Shape:

(3,128,128)

Mask Classes:

0 = Non-Water

1 = Water

In [64]:
bad_images = 0

for file in train_images:

    image = np.load(file)

    if np.isnan(image).any():
        bad_images += 1

print("Images Containing NaN:", bad_images)

Images Containing NaN: 0


## Notebook Summary

This notebook preprocesses the STRUM Sentinel-2 flood mapping dataset for training segmentation models such as U-Net and the Hybrid SAM-U-Net architecture. The workflow includes dataset validation, train-validation-test splitting, RGB band extraction, image normalization, mask conversion, and saving the processed data in NumPy format for efficient model training.

### Dataset Overview

| Property                 | Value     |
| ------------------------ | --------- |
| Total Images             | 2,675     |
| Total Masks              | 2,675     |
| Matched Image-Mask Pairs | 2,675     |
| Original Image Size      | 128 × 128 |
| Original Bands           | 9         |
| Extracted Bands          | 3 (RGB)   |

### Dataset Split

| Split      |   Samples | Percentage |
| ---------- | --------: | ---------: |
| Train      |     2,140 |        80% |
| Validation |       267 |        10% |
| Test       |       268 |        10% |
| **Total**  | **2,675** |   **100%** |

### Preprocessing Steps

1. Verified dataset structure and image-mask correspondence.
2. Created reproducible train/validation/test splits using a fixed random seed.
3. Extracted RGB channels from the original 9-band Sentinel-2 imagery.
4. Applied min-max normalization to scale pixel values into the range [0, 1].
5. Converted multiclass flood masks into binary segmentation masks:

   * 0 → Background / Non-flood
   * All other classes → Flood
6. Saved processed images and masks as NumPy arrays for faster loading during training.

### Image Normalization Example

| Metric        | Before Normalization | After Normalization |
| ------------- | -------------------: | ------------------: |
| Minimum Value |               0.2033 |                 0.0 |
| Maximum Value |               1.5467 |                 1.0 |

### Original Mask Class Distribution

| Class | Pixel Count |
| ----- | ----------: |
| 0     |  31,739,964 |
| 1     |   9,865,084 |
| 2     |     570,702 |
| 3     |     853,493 |
| 4     |     141,386 |
| 5     |     656,571 |

### Processed Data Verification

| Check                   | Result        |
| ----------------------- | ------------- |
| Train Images Saved      | 2,140         |
| Train Masks Saved       | 2,140         |
| Validation Images Saved | 267           |
| Validation Masks Saved  | 267           |
| Test Images Saved       | 268           |
| Test Masks Saved        | 268           |
| Image Shape             | (3, 128, 128) |
| Mask Shape              | (128, 128)    |
| Image Range             | 0.0 – 1.0     |
| Mask Classes            | [0, 1]        |
| Images Containing NaN   | 0             |

## Conclusion

The STRUM dataset was successfully preprocessed and converted into a format suitable for deep learning-based flood segmentation. RGB images were normalized to a consistent range, multiclass masks were converted into binary flood masks, and the dataset was split into training, validation, and testing subsets following an 80-10-10 ratio. Final verification confirmed correct image-mask alignment, valid pixel ranges, binary mask generation, and the absence of NaN values, ensuring the dataset is ready for model training and evaluation.
